# AverageField Hardware FIFO Sanity

This notebook opens the Spectrum digitizer, loads the freshly built `AverageField` module from `build\\windows-qom-ninja`, creates `afw`, and leaves the acquisition cell separate so external triggering can be started manually first.

Current native `AverageField` processing expects four physical Spectrum channels packed as two complex fields: `[0, 1] -> field 1`, `[2, 3] -> field 2`. Do not switch this notebook to `[0, 1]` only until the explicit 2-channel native mode is implemented.

In [ ]:
from pathlib import Path
import ctypes
import importlib.util
import os
import sys
import time

import numpy as np

PROJECT = Path.cwd()
if not (PROJECT / "CMakePresets.json").exists() and Path(r"C:\Users\Qop\AverageField").exists():
    PROJECT = Path(r"C:\Users\Qop\AverageField")

QO_REPO = Path(r"C:\Users\Qop\QO-measurements")
if QO_REPO.exists() and str(QO_REPO) not in sys.path:
    sys.path.insert(0, str(QO_REPO))

if hasattr(os, "add_dll_directory"):
    cuda_path = Path(os.environ.get("CUDA_PATH", r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v13.0"))
    for dll_dir in (cuda_path / "bin" / "x64", cuda_path / "bin", Path(r"C:\Windows\System32")):
        if dll_dir.exists():
            os.add_dll_directory(str(dll_dir))

from drivers.Spectrum_m4x import SPCM, SPCM_MODE, SPCM_TRIGGER


def load_averagefield(build_dir=PROJECT / "build" / "windows-qom-ninja"):
    build_dir = Path(build_dir)
    candidates = sorted(build_dir.glob("AverageField*.pyd"))
    candidates += sorted((build_dir / "Release").glob("AverageField*.pyd"))
    if not candidates:
        raise FileNotFoundError(f"No AverageField*.pyd found in {build_dir}")

    module_path = candidates[0]
    spec = importlib.util.spec_from_file_location("AverageField", module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not create import spec for {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    print("Loaded:", module_path)
    return module


AverageField = load_averagefield()

In [ ]:
DIGITIZER = b"/dev/spcm0"
USE_EXTERNAL_CLOCK = True

dig = SPCM(DIGITIZER)
if USE_EXTERNAL_CLOCK:
    dig.setup_external_clock()

dig_params = {
    "channels": [0, 1, 2, 3],
    "ch_amplitude": 200,
    "dur_seg": 1000,
    "n_avg": 0,
    "n_seg": 1 << 10,
    "oversampling_factor": 1,
    "pretrigger": 32,
    "digitizer_delay": 90,
    "mode": SPCM_MODE.MULTIPLE_FIFO,
    "trig_source": SPCM_TRIGGER.EXT0,
}

dig.set_parameters(dig_params)

print("channels:", dig.channels)
print("sample_rate:", dig.get_sample_rate())
print("segment_size:", dig.get_segment_size())
print("n_seg:", dig.n_seg)
print("driver _bufsize:", getattr(dig, "_bufsize", None))
try:
    print("estimated MiB/s at 1000 ns period:", dig.calc_transfer_speed_mib(1000))
except Exception as exc:
    print("transfer estimate unavailable:", exc)

In [ ]:
averages = 1 << 10
batch = int(dig_params["n_seg"])
part = 1.0
second_oversampling = 1

afw = AverageField.AverageFieldMeasurer(
    ctypes.addressof(dig.h_card.contents),
    averages,
    batch,
    part,
    second_oversampling,
)

afw.set_amplitude(int(dig_params["ch_amplitude"]))
afw.set_calibration(0, 1.0, 0.0, 0.0, 0.0)
afw.set_calibration(1, 1.0, 0.0, 0.0, 0.0)
afw.set_firwin(-512.0, 512.0)
afw.set_intermediate_frequency(0.0)

print("afw created")
print("total_length:", afw.get_total_length())
print("trace_length:", afw.get_trace_length())
print("resampled_trace_length:", afw.get_resampled_trace_length())
print("out_size:", afw.get_out_size())
print("notify_size:", afw.get_notify_size())

Start the external trigger/output sequence before running the next cell.

In [ ]:
afw.reset_output()
t0 = time.perf_counter()
afw.measure()
elapsed = time.perf_counter() - t0
print(f"measure() elapsed: {elapsed:.3f} s for averages={averages}, batch={batch}")

In [ ]:
avg0, avg1 = afw.get_average_field()
avg0 = np.asarray(avg0)
avg1 = np.asarray(avg1)
s21 = afw.get_s21()
g1 = np.asarray(afw.get_g1_correlator())
g1_diag = np.diag(g1)

print("average field shapes:", avg0.shape, avg1.shape)
print("s21:", s21)
print("g1 shape:", g1.shape)
print("finite avg0/avg1/g1:", np.isfinite(avg0).all(), np.isfinite(avg1).all(), np.isfinite(g1).all())
print("avg0 first 5:", avg0[:5])
print("avg1 first 5:", avg1[:5])
print("g1 diagonal first 5:", g1_diag[:5])

In [ ]:
# Run when done.
try:
    afw.free()
except Exception as exc:
    print("afw.free() failed:", exc)

try:
    dig.close()
except Exception as exc:
    print("dig.close() failed:", exc)